# Ground Truth Token Counts

`src/evaluation/ground_truth.py`의 질문과 정답 토큰 수를 계산합니다.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src" / "evaluation" / "ground_truth.py").exists():
    project_root = project_root.parent

sys.path.append(str(project_root))
project_root

In [ ]:
import pandas as pd
import yaml
from transformers import AutoTokenizer

from src.evaluation.ground_truth import make_ground_truth_dataframe

In [ ]:
config_path = project_root / "configs" / "experiments" / "bge-m3_qwen3-8B.yaml"

with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

tokenizer_model = config["generation"]["model"]
tokenizer_model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_model,
    trust_remote_code=True,
)

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(str(text), add_special_tokens=False))

In [ ]:
df = make_ground_truth_dataframe()

df["question_tokens"] = df["question"].apply(count_tokens)
df["ground_truth_tokens"] = df["ground_truth"].apply(count_tokens)
df["qa_tokens"] = df["question_tokens"] + df["ground_truth_tokens"]

df.head()

In [ ]:
summary = (
    df.groupby(["document_id", "document_name"], as_index=False)
    .agg(
        row_count=("question_id", "count"),
        question_tokens_total=("question_tokens", "sum"),
        question_tokens_mean=("question_tokens", "mean"),
        question_tokens_max=("question_tokens", "max"),
        ground_truth_tokens_total=("ground_truth_tokens", "sum"),
        ground_truth_tokens_mean=("ground_truth_tokens", "mean"),
        ground_truth_tokens_max=("ground_truth_tokens", "max"),
        qa_tokens_total=("qa_tokens", "sum"),
        qa_tokens_mean=("qa_tokens", "mean"),
        qa_tokens_max=("qa_tokens", "max"),
    )
)

summary

In [ ]:
overall = df[["question_tokens", "ground_truth_tokens", "qa_tokens"]].agg(
    ["count", "sum", "mean", "median", "min", "max"]
)

overall

In [ ]:
output_dir = project_root / "outputs" / "analysis"
output_dir.mkdir(parents=True, exist_ok=True)

detail_path = output_dir / "ground_truth_token_counts.csv"
summary_path = output_dir / "ground_truth_token_summary.csv"

df.to_csv(detail_path, index=False, encoding="utf-8-sig")
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

detail_path, summary_path

In [ ]:
import ast

def make_qa_prompt(joined_contexts: str, question: str) -> str:
    return (
        "당신은 제안서 분석 전문가입니다. 주어진 문맥에만 철저히 기반하여 질문에 답하세요.\n"
        "문맥에 없는 내용이거나 확인 불가능한 정보라면 솔직하게 '문맥상 확인할 수 없습니다'라고 답하세요.\n\n"
        f"[문맥]:\n{joined_contexts}\n\n[질문]:\n{question}"
    )

def join_retrieved_contexts(value) -> str:
    if isinstance(value, list):
        return "\n\n".join(str(context) for context in value)
    if pd.isna(value):
        return ""

    try:
        parsed_value = ast.literal_eval(str(value))
    except (SyntaxError, ValueError):
        return str(value)

    if isinstance(parsed_value, list):
        return "\n\n".join(str(context) for context in parsed_value)
    return str(parsed_value)

generation_results_path = project_root / config["output"]["generation_eval_results"].format(
    strategy_name=config["output"]["strategy_name"]
)

if not generation_results_path.exists():
    generation_candidates = sorted(
        (project_root / "outputs" / "evaluation").glob("*generation_results.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not generation_candidates:
        raise FileNotFoundError("outputs/evaluation/*generation_results.csv 파일이 없습니다.")
    generation_results_path = generation_candidates[0]

prompt_df = pd.read_csv(generation_results_path)

if "retrieved_context" in prompt_df.columns:
    prompt_df["joined_contexts"] = prompt_df["retrieved_context"].fillna("").astype(str)
else:
    prompt_df["joined_contexts"] = prompt_df["retrieved_contexts"].apply(join_retrieved_contexts)

prompt_df["qa_prompt"] = prompt_df.apply(
    lambda row: make_qa_prompt(row["joined_contexts"], row["question"]),
    axis=1,
)
prompt_df["qa_prompt_tokens"] = prompt_df["qa_prompt"].apply(count_tokens)

prompt_token_counts = prompt_df[
    ["document_id", "document_name", "question_id", "qa_prompt_tokens"]
]

prompt_summary = prompt_token_counts["qa_prompt_tokens"].agg(
    ["count", "sum", "mean", "median", "min", "max"]
)

qa_prompt_token_counts_path = output_dir / "qa_prompt_token_counts.csv"
prompt_token_counts.to_csv(qa_prompt_token_counts_path, index=False, encoding="utf-8-sig")

generation_results_path, qa_prompt_token_counts_path, prompt_token_counts, prompt_summary